### Install Dependencies

In [1]:
import pandas as pd
import openai

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

### Read Sample dataset from Amazon

In [3]:
df_items = pd.read_json("../../data/meta_Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", lines=True)

In [4]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,AMAZON FASHION,"Bosttor Bluetooth Beanie Hat with Light, Headl...",4.5,2342,"[100% Acrylic, Elastic closure, Hand Wash Only...",[],26.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Beanie Hat with Bluetooth Headphon...,Bosttor,"[Electronics, Headphones, Earbuds & Accessorie...","{'Department': 'unisex-adult', 'Date First Ava...",B0BH41HYFZ,NaN,NaN,NaN
1,All Electronics,"Aceele USB and USB C to Ethernet Adapter, 3.3f...",4.2,148,[【USB or USB C to Ethernet Adapter】The Etherne...,[],14.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],Aceele,"[Electronics, Computers & Accessories, Network...",{'Product Dimensions': '3.54 x 0.98 x 0.71 inc...,B0BKPB2YQ9,NaN,NaN,NaN
2,All Electronics,HDMI Switch 3 in 1 Out 4K UHD HDMI Switcher Sp...,4.2,3331,[📍3-Port HDMI Switch: This aluminum HDMI switc...,[],18.98,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Darren reviews VWRHAR 3-1 splitter...,VWRHar,"[Electronics, Home Audio, Home Audio Accessori...",{'Package Dimensions': '6.1 x 4.21 x 0.94 inch...,B09MM5QT3R,NaN,NaN,NaN
3,All Electronics,"Smart Watch,Ip67 Waterproof Bluetooth Smartwat...",3.4,125,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Burxoe,[],{'Package Dimensions': '3.15 x 3.07 x 2.52 inc...,B09Q5TNDHY,NaN,NaN,NaN
4,Cell Phones & Accessories,"(3 Pack) Cute Airpod Case for Airpods 2&1,3D D...",4.7,335,[【Compatible Airpods 1/2 】This case designed f...,[1],11.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'airpod cases', 'url': 'https://www...",UGUHY,"[Electronics, Headphones, Earbuds & Accessorie...",{'Package Dimensions': '7.44 x 4.61 x 1.57 inc...,B0BZJKX7MS,NaN,NaN,NaN


In [5]:
### Preprocess title and features

def preprocess_description(row):
    return f"{row['title']} {' '.join(row['features'])}"

In [6]:
def extract_first_large_image(row):
    return row['images'][0].get('large', '')

In [7]:
df_items['preprocessed_description'] = df_items.apply(preprocess_description, axis=1)
df_items['image_url'] = df_items.apply(extract_first_large_image, axis=1)

In [8]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author,preprocessed_description,image_url
0,AMAZON FASHION,"Bosttor Bluetooth Beanie Hat with Light, Headl...",4.5,2342,"[100% Acrylic, Elastic closure, Hand Wash Only...",[],26.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Beanie Hat with Bluetooth Headphon...,Bosttor,"[Electronics, Headphones, Earbuds & Accessorie...","{'Department': 'unisex-adult', 'Date First Ava...",B0BH41HYFZ,NaN,NaN,NaN,"Bosttor Bluetooth Beanie Hat with Light, Headl...",https://m.media-amazon.com/images/I/51b7qcj8dZ...
1,All Electronics,"Aceele USB and USB C to Ethernet Adapter, 3.3f...",4.2,148,[【USB or USB C to Ethernet Adapter】The Etherne...,[],14.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],Aceele,"[Electronics, Computers & Accessories, Network...",{'Product Dimensions': '3.54 x 0.98 x 0.71 inc...,B0BKPB2YQ9,NaN,NaN,NaN,"Aceele USB and USB C to Ethernet Adapter, 3.3f...",https://m.media-amazon.com/images/I/41WsGRr-3T...
2,All Electronics,HDMI Switch 3 in 1 Out 4K UHD HDMI Switcher Sp...,4.2,3331,[📍3-Port HDMI Switch: This aluminum HDMI switc...,[],18.98,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Darren reviews VWRHAR 3-1 splitter...,VWRHar,"[Electronics, Home Audio, Home Audio Accessori...",{'Package Dimensions': '6.1 x 4.21 x 0.94 inch...,B09MM5QT3R,NaN,NaN,NaN,HDMI Switch 3 in 1 Out 4K UHD HDMI Switcher Sp...,https://m.media-amazon.com/images/I/41fsjaknf1...
3,All Electronics,"Smart Watch,Ip67 Waterproof Bluetooth Smartwat...",3.4,125,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Burxoe,[],{'Package Dimensions': '3.15 x 3.07 x 2.52 inc...,B09Q5TNDHY,NaN,NaN,NaN,"Smart Watch,Ip67 Waterproof Bluetooth Smartwat...",https://m.media-amazon.com/images/I/51YFypeuZh...
4,Cell Phones & Accessories,"(3 Pack) Cute Airpod Case for Airpods 2&1,3D D...",4.7,335,[【Compatible Airpods 1/2 】This case designed f...,[1],11.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'airpod cases', 'url': 'https://www...",UGUHY,"[Electronics, Headphones, Earbuds & Accessorie...",{'Package Dimensions': '7.44 x 4.61 x 1.57 inc...,B0BZJKX7MS,NaN,NaN,NaN,"(3 Pack) Cute Airpod Case for Airpods 2&1,3D D...",https://m.media-amazon.com/images/I/41V7Wi2ckb...


In [9]:
### Sample items from dataset
df_sample = df_items.sample(n=50, random_state=7)

In [10]:
df_data_to_embed = df_sample[['preprocessed_description', 'image_url', 'rating_number', 'price', 'average_rating', 'parent_asin', ]]

In [11]:
data_to_embed = df_data_to_embed.to_dict(orient = "records")

In [12]:
data_to_embed

[{'preprocessed_description': 'MANDDDWU 80x100 Monocular-Telescope Low Night Vision Monoculars High Definition for Adults High Powered with Smartphone Adapter Telescope Hunting Wildlife Bird Watching Travel Camping Hiking ',
  'image_url': 'https://m.media-amazon.com/images/I/51h4NUuSazL._AC_.jpg',
  'rating_number': 446,
  'price': 29.99,
  'average_rating': 3.8,
  'parent_asin': 'B09QQ2J8XN'},
 {'preprocessed_description': 'Car Windshield Dashboard Tablet Holder for 5.5-12.9" Tablet Smartphone, EXSHOW Long Arm Suction Cup Car Window Tablet Cell Phone Mount for iPad Pro Air Mini, Samsung Galaxy Tab, iPhone, Xiaomi etc ',
  'image_url': 'https://m.media-amazon.com/images/I/41k1INUbfAL._AC_.jpg',
  'rating_number': 103,
  'price': nan,
  'average_rating': 4.0,
  'parent_asin': 'B08DLWKXGL'},
 {'preprocessed_description': 'SOOMFON Bluetooth 5.0 FM Transmitter for Car, Stronger Microphone & Bass Sound Bluetooth Radio Transmitter Car Adapter, Support 20W PD+QC3.0, 7 Colors LED Backlit, Wir

### Define embedding function

In [13]:
from google import genai
from google.genai import types
import os

gemini_client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
result = gemini_client.models.embed_content(
    model="gemini-embedding-001",
    contents="random text",
    config=types.EmbedContentConfig(task_type="SEMANTIC_SIMILARITY")
)

In [25]:
len(result.embeddings[0].values)

3072

In [ ]:
### Other task_type = SEMANTIC_SIMILARITY, CLASSIFICATION, CLUSTERING, RETRIEVAL_DOCUMENT, CODE_RETRIEVAL_QUERY, QUESTION_ANSWERING, FACT_VERIFICATION

def get_embeddings(text, task_type="SEMANTIC_SIMILARITY", model="gemini-embedding-001"):
    result = gemini_client.models.embed_content(
        model=model,
        contents=text,
        config=types.EmbedContentConfig(task_type=task_type)
    )
    return result.embeddings[0].values


In [27]:
get_embeddings("random text")

[-0.01684018,
 0.018565726,
 0.0068769087,
 -0.06927192,
 -0.00553585,
 0.010261666,
 0.011438973,
 0.004223627,
 0.012219261,
 0.0132948095,
 0.0062436694,
 -0.010938613,
 -0.021939095,
 0.04920108,
 0.11680988,
 0.0032733437,
 -0.02397843,
 0.023174794,
 0.002537348,
 -0.018138312,
 -0.0015183971,
 0.0076536317,
 -0.0009793843,
 -0.015984554,
 -0.015191916,
 -0.017045336,
 0.029863806,
 0.016304195,
 0.02727329,
 -0.0073262565,
 -0.005536062,
 0.033683773,
 0.0033344007,
 0.00010679571,
 0.013984189,
 0.01484251,
 0.022953782,
 0.016573895,
 0.0063176355,
 0.046974257,
 -0.0121840425,
 -3.0958377e-05,
 0.00045738928,
 -0.0030470006,
 0.02308657,
 0.011519489,
 0.004947156,
 -0.007096894,
 0.01923382,
 0.035404667,
 -0.014046564,
 -0.009017905,
 0.004565115,
 -0.15831394,
 -0.0008534757,
 0.0087582795,
 -0.006312307,
 0.0020324066,
 0.012475935,
 0.012627499,
 -0.030367404,
 0.026159406,
 -0.030088518,
 -0.027111536,
 0.01753244,
 -0.022347193,
 0.008908948,
 -0.0077574877,
 -0.040777

#### Create Qdrant Collection

In [29]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [30]:
qdrant_client.create_collection(
    collection_name="Amazon-items-collection-01",
    vectors_config=VectorParams(size=3072, distance=Distance.COSINE),
)

True

In [32]:
### Embed Data

pointstruct = PointStruct(
    id=0,
    vector=get_embeddings("Hello World"),
    payload={
        "text": "Hello World",
        "model": "gemini-embedding-001"
    }
)

In [34]:
pointstruct

PointStruct(id=0, vector=[-0.030313896, -0.0011937966, 0.011944085, -0.06331687, -0.018562978, 0.006006837, -0.013848489, -0.0069041, 0.018909177, 0.0067635323, -0.0013741076, -0.012544641, -0.012370406, 0.05024206, 0.14017393, -0.0023973074, -0.0021481998, -0.006960438, 0.008675419, -0.012715995, -0.001812732, 0.0038445718, 0.007866856, -0.018134868, -0.024915073, -0.012399078, 0.025578758, -0.002082866, 0.028287888, -0.003586746, 0.004083515, 0.015775781, -0.0004313633, 0.008374812, 0.008510412, 0.017772533, 0.02911796, 0.020505413, 0.00085054873, 0.027917791, -0.0065016323, -0.00055489736, 0.010354559, -0.014056266, 0.006188762, 0.01914849, 0.0056114774, -0.02003173, 0.009173776, 0.046414502, -0.028422333, 0.0009706299, 0.00034367744, -0.15495303, -0.03205103, 0.015597616, 0.0014483529, 0.013244147, -0.0012753017, -0.0069762473, -0.0065743234, 0.026333023, -0.033914164, -0.03351515, -0.014084813, -0.021122253, -0.0037559562, -0.015894454, -0.0132044535, 0.0009266045, -0.0046076463, 

### Amazon Data

In [35]:
len(data_to_embed)

50

In [36]:
pointstructs=[]
for i, data in enumerate(data_to_embed):
    embedding = get_embeddings(data["preprocessed_description"])
    pointstructs.append(
        PointStruct(
            id=i,
            vector=embedding,
            payload=data
        )
    )

In [37]:
pointstructs

[PointStruct(id=0, vector=[-0.010977576, -0.008538937, 0.004236455, -0.07589605, -0.0014879254, 0.0053786193, 0.005274366, 0.008663739, 0.009658452, 0.004588908, 0.0051917583, -0.014114624, -0.03356956, 0.019495793, 0.11103883, -0.0090452675, -0.0020489923, 0.005078651, 0.0126835825, 0.009409637, -0.023331175, -0.010531944, 0.0142383715, -0.021991448, -0.01286905, -0.0106603755, 0.023119094, 0.020531483, 0.039112955, -0.0022672636, 0.00088329794, 0.022663286, 0.0046168887, 0.008672598, -0.00349664, -0.0077687493, 0.017381888, 0.019098476, 0.008272457, 0.024488414, -0.0024776082, -0.015764402, 0.0025611836, 0.013940097, 0.016249174, 0.009614298, -0.006651214, -0.026154011, 0.029669905, 0.03881756, -0.0025897012, 0.008393907, 0.020003777, -0.1632329, 0.009418588, -0.0028868266, -0.014461994, -7.6997596e-05, 0.008279342, -0.0023833527, -0.02737789, 0.035969835, 0.008231631, -0.015232494, -0.020477835, -0.04626063, -0.01682074, -0.0111779645, -0.020411115, -0.0155845275, -0.0063450383, -0.

In [38]:
len(pointstructs)

50

In [39]:
### Write embedded data to Qdrant

qdrant_client.upsert(
    collection_name="Amazon-items-collection-01",
    points=pointstructs
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

### Define a function for data retrieval

In [40]:
def retrieve_data(query, k=5):
    query_embedding = get_embeddings(query)
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )
    return results

In [42]:
retrieve_data("do you have any smart watches?").points

[ScoredPoint(id=5, version=1, score=0.79831743, payload={'preprocessed_description': "Smart Watch, Tykoit Smartwatch 41mm with Heart Rate Monitor & Sleep Tracking & SpO2, IP68 Waterproof Fitness Tracker with Step/Calorie Counter Compatible with Android and iOS Phones Heart Rate, Sleep and SpO2 Monitor: This smart watch built in high performance motion sensor automatically detects your heart rate in real time, and also measure your SpO2 and deep, light sleep state which can help you better understand your health. (NOTE: the data cannot be used as a medical-grade test) All-day Activity Tracking: This smartwatch has 8 kinds of sports modes (walking, running, treadmill, bicycle, yoga and other workouts). The fitness tracker display shows your accurate steps, distance, calories, active minutes and you can also see more detailed data on App. Customizable Watch Face: Our smartwatches support customizable watch faces with your favorite pictures to suit your daily mood and occasion. And the fit